# Gold Layer — Star Schema & Aggregated KPIs

| Item | Detail |
|---|---|
| **Source** | Silver Delta tables in `maven_market_uc.silver` |
| **Target** | Gold Delta tables in `maven_market_uc.gold` |
| **Run Type** | DLT Pipeline — Target catalog: `maven_market_uc`, Target schema: `gold` |
| **Pattern** | Star Schema (fact + dimensions) + Pre-aggregated KPI tables |

### Gold Tables
| Table | Type | Description |
|---|---|---|
| `dim_customers` | Dimension | Customer attributes (current SCD2 snapshot) |
| `dim_products` | Dimension | Product attributes with price tier |
| `dim_stores` | Dimension | Store + region attributes |
| `dim_calendar` | Dimension | Date dimension |
| `fact_sales` | Fact | Transactions joined with product prices |
| `fact_returns` | Fact | Returns joined with product prices |
| `agg_daily_sales` | Aggregate | Daily sales by store |
| `agg_monthly_sales` | Aggregate | Monthly sales by store + product brand |
| `agg_customer_lifetime` | Aggregate | Customer lifetime value |
| `agg_product_performance` | Aggregate | Product performance metrics |
| `agg_store_performance` | Aggregate | Store performance with return rates |
| `kpi_executive_summary` | KPI | High-level executive metrics |

## Imports & Helper

In [0]:
import dlt
from pyspark.sql.functions import (
    col, sum as _sum, count, countDistinct,
    avg, min as _min, max as _max,
    round as spark_round, when, lit,
    current_timestamp, concat, date_format,
    first, coalesce, window
)
from pyspark.sql.types import (
    IntegerType, DoubleType
)

---
## Dimension Tables

dim_customers — Current snapshot from SCD2

In [0]:
@dlt.table(
    name="maven_catalog.gold_schema.dim_customers",
    comment="Customer dimension — current records only from SCD2",
    table_properties={"quality": "gold"}
)
def dim_customers():
    return (
        dlt.read("maven_catalog.silver_schema.slv_customers")
        .filter(col("__END_AT").isNull())
        .select(
            col("customer_id").alias("customer_key"),
            "customer_id",
            "customer_acct_num",
            "full_name",
            "customer_city",
            "customer_state_province",
            "customer_country",
            "birthdate",
            "age",
            "marital_status",
            "yearly_income",
            "income_band_lower",
            "gender",
            "total_children",
            "num_children_at_home",
            "education",
            "member_card",
            "occupation",
            "homeowner",
        )
    )

dim_products

In [0]:
@dlt.table(
    name="maven_catalog.gold_schema.dim_products",
    comment="Product dimension with price tier and margin",
    table_properties={"quality": "gold"}
)
def dim_products():
    return (
        dlt.read("maven_catalog.silver_schema.slv_products")
        .select(
            col("product_id").alias("product_key"),
            "product_id",
            "product_brand",
            "product_name",
            "product_sku",
            "product_retail_price",
            "product_cost",
            "product_weight",
            "profit_margin",
            "profit_amount",
            "price_tier",
            "recyclable",
            "low_fat",
        )
    )

dim_stores

In [0]:
@dlt.table(
    name="maven_catalog.gold_schema.dim_stores",
    comment="Customer dimension — cudim_stores",
    #comment="Store dimension with region and area metrics",
    table_properties={"quality": "gold"}
)
def dim_stores():
    return (
        dlt.read("maven_catalog.silver_schema.slv_stores")
        .select(
            col("store_id").alias("store_key"),
            "store_id",
            "store_name",
            "store_type",
            "store_city",
            "store_state",
            "store_country",
            "sales_district",
            "sales_region",
            "store_age_years",
            "years_since_remodel",
            "total_sqft",
            "grocery_sqft",
            "non_grocery_sqft",
            "grocery_pct",
        )
    )

dim_calendar

In [0]:
@dlt.table(
    name="maven_catalog.gold_schema.dim_calendar",
    comment="Date dimension for time-based analysis",
    table_properties={"quality": "gold"}
)
def dim_calendar():
    return (
        dlt.read("maven_catalog.silver_schema.slv_calendar")
        .select(
            col("date").alias("date_key"),
            "date",
            "year",
            "quarter",
            "month",
            "month_name",
            "month_short",
            "week_of_year",
            "day_of_month",
            "day_of_week",
            "day_name",
            "is_weekend",
            "is_weekday",
            "fiscal_year",
            "fiscal_quarter",
            "year_month",
        )
    )

---
## Fact Tables

fact_sales — Transactions enriched with product prices

In [0]:
@dlt.table(
    name="maven_catalog.gold_schema.fact_sales",
    comment="Sales fact table — transactions joined with product pricing",
    table_properties={"quality": "gold"}
)
def fact_sales():
    transactions = dlt.read("maven_catalog.silver_schema.slv_transactions")
    products = (
        dlt.read("maven_catalog.silver_schema.slv_products")
        .select(
            "product_id",
            "product_retail_price",
            "product_cost",
        )
    )

    return (
        transactions
        .join(products, "product_id", "left")
        .withColumn(
            "total_revenue",
            spark_round(
                col("quantity") * col("product_retail_price"), 2
            )
        )
        .withColumn(
            "total_cost",
            spark_round(
                col("quantity") * col("product_cost"), 2
            )
        )
        .withColumn(
            "total_profit",
            spark_round(
                col("total_revenue") - col("total_cost"), 2
            )
        )
        .select(
            "transaction_date",
            "stock_date",
            "product_id",
            "customer_id",
            "store_id",
            "quantity",
            "product_retail_price",
            "product_cost",
            "total_revenue",
            "total_cost",
            "total_profit",
        )
    )

fact_returns — Returns enriched with product prices

In [0]:
@dlt.table(
    name="maven_catalog.gold_schema.fact_returns",
    comment="Returns fact table — returns joined with product pricing",
    table_properties={"quality": "gold"}
)
def fact_returns():
    returns = dlt.read("maven_catalog.silver_schema.slv_returns")
    products = (
        dlt.read("maven_catalog.silver_schema.slv_products")
        .select(
            "product_id",
            "product_retail_price",
            "product_cost",
        )
    )

    return (
        returns
        .join(products, "product_id", "left")
        .withColumn(
            "return_revenue_loss",
            spark_round(
                col("quantity") * col("product_retail_price"), 2
            )
        )
        .withColumn(
            "return_cost_loss",
            spark_round(
                col("quantity") * col("product_cost"), 2
            )
        )
        .select(
            "return_date",
            "product_id",
            "store_id",
            "quantity",
            "product_retail_price",
            "product_cost",
            "return_revenue_loss",
            "return_cost_loss",
        )
    )

In [0]:
@dlt.table(
    name="maven_catalog.gold_schema.fact_kafka_orders",
    comment="Real-time order facts joined with dimensions",
    table_properties={"quality": "gold"}
)
def fact_kafka_orders():
    orders = dlt.read("slv_kafka_orders").alias("o")
    products = dlt.read("dim_products").alias("p")
    
    return (
        orders.join(products, col("o.product_id") == col("p.product_key"), "left")
        .select(
            col("o.order_id"),
            col("o.event_time"),
            col("o.product_id").alias("product_key"),
            col("o.customer_id").alias("customer_key"),
            col("o.store_id").alias("store_key"),
            col("o.quantity"),
            col("o.total_amount").alias("revenue"),
            col("o.unit_price"),
            col("o.order_size"),
            # Add profit calculation using product cost from dimension
            spark_round(col("o.total_amount") - (col("o.quantity") * col("p.product_cost")), 2).alias("estimated_profit")
        )
    )

In [0]:
@dlt.table(
    name="maven_catalog.gold_schema.fact_inventory_movements",
    comment="Inventory state changes tracked over time",
    table_properties={"quality": "gold"}
)
def fact_inventory_movements():
    inventory = dlt.read("slv_kafka_inventory").alias("i")
    
    return (
        inventory.select(
            col("i.event_time"),
            col("i.product_id").alias("product_key"),
            col("i.store_id").alias("store_key"),
            "current_stock",
            "quantity_change",
            "change_type",
            "stock_status",
            "is_low_stock",
            "is_restock"
        )
    )

---
## Aggregate Tables

agg_daily_sales — Daily sales by store

In [0]:
@dlt.table(
    name="maven_catalog.gold_schema.agg_daily_sales",
    comment="Daily sales aggregated by store — built from Fact and Dim tables",
    table_properties={"quality": "gold"}
)
def agg_daily_sales():
    # Read from Gold Facts and Dimensions instead of Silver
    sales = dlt.read("fact_sales") 
    stores = dlt.read("dim_stores")

    return (
        sales
        .groupBy("transaction_date", "store_id")
        .agg(
            _sum("quantity").alias("total_units"),
            _sum("total_revenue").alias("total_revenue"),
            _sum("total_cost").alias("total_cost"),
            _sum("total_profit").alias("total_profit"),
            countDistinct("product_id").alias("unique_products"),
            countDistinct("customer_id").alias("unique_customers"),
            count("*").alias("total_transactions")
        )
        .join(stores.select("store_key", "store_name", "sales_region"), 
              col("store_id") == col("store_key"), "left")
        .withColumn(
            "avg_transaction_value",
            spark_round(col("total_revenue") / col("total_transactions"), 2)
        )
    )

agg_monthly_sales — Monthly by store + brand

In [0]:
@dlt.table(
    name="maven_catalog.gold_schema.agg_monthly_sales",
    comment="Monthly sales trends — built from Fact and Dim tables",
    table_properties={"quality": "gold"}
)
def agg_monthly_sales():
    sales = dlt.read("fact_sales")
    products = dlt.read("dim_products")
    calendar = dlt.read("dim_calendar")

    return (
        sales
        .join(calendar, sales.transaction_date == calendar.date_key, "left")
        .join(products.select("product_key", "product_brand", "price_tier"), 
              sales.product_id == products.product_key, "left")
        .groupBy(
            "year", "month", "month_name", "quarter",
            "year_month", "store_id", "product_brand", "price_tier"
        )
        .agg(
            _sum("quantity").alias("total_units"),
            _sum("total_revenue").alias("total_revenue"),
            _sum("total_cost").alias("total_cost"),
            _sum("total_profit").alias("total_profit"),
            countDistinct("customer_id").alias("unique_customers"),
            count("*").alias("total_transactions")
        )
    )

agg_product_performance — per product

In [0]:
@dlt.table(
    name="maven_catalog.gold_schema.agg_store_performance",
    comment="Store performance metrics — built from Fact and Dim tables",
    table_properties={"quality": "gold"}
)
def agg_store_performance():
    sales_fact = dlt.read("fact_sales")
    returns_fact = dlt.read("fact_returns")
    stores_dim = dlt.read("dim_stores")

    # Aggregate and Alias
    sales_agg = sales_fact.groupBy("store_id").agg(
        _sum("quantity").alias("total_units_sold"),
        _sum("total_revenue").alias("total_revenue"),
        _sum("total_cost").alias("total_cost"),
        _sum("total_profit").alias("total_profit"),
        count("*").alias("total_transactions")
    ).alias("sales_agg") # <--- Explicitly naming the result

    returns_agg = returns_fact.groupBy("store_id").agg(
        _sum("quantity").alias("total_units_returned")
    ).alias("returns_agg")

    # Drop duplicate store_id and Alias
    stores_info = stores_dim.drop("store_id").alias("stores_info")

    return (
        sales_agg
        .join(returns_agg, "store_id", "left")
        # Now "sales_agg.store_id" is a valid reference
        .join(stores_info, col("sales_agg.store_id") == col("stores_info.store_key"), "left")
        .withColumn("total_units_returned", coalesce(col("total_units_returned"), lit(0)))
        .withColumn("return_rate", spark_round(col("total_units_returned") / col("total_units_sold") * 100, 2))
        .withColumn("revenue_per_sqft", spark_round(col("total_revenue") / col("total_sqft"), 2))
    )

Orders per Minute (Throughput Trend)

In [0]:
@dlt.table(
    name="maven_catalog.gold_schema.agg_kafka_throughput_min",
    comment="Operational metric: Orders per minute"
)
def agg_kafka_throughput_min():
    return (
        dlt.read("fact_kafka_orders")
        .groupBy(window(col("event_time"), "1 minute").alias("time_window"))
        .agg(
            count("order_id").alias("order_count"),
            _sum("revenue").alias("total_revenue")
        )
        .select(
            col("time_window.start").alias("minute_bin"),
            "order_count",
            "total_revenue"
        )
    )

Inventory Alert Summary

In [0]:
@dlt.table(
    name="maven_catalog.gold_schema.agg_inventory_alerts",
    comment="Current products requiring immediate operational attention"
)
def agg_inventory_alerts():
    inventory = dlt.read("fact_inventory_movements")
    stores = dlt.read("dim_stores")
    products = dlt.read("dim_products")
    
    return (
        inventory.filter(col("stock_status").isin("Out of Stock", "Low Stock"))
        .join(stores, inventory.store_key == stores.store_key, "left")
        .join(products, inventory.product_key == products.product_key, "left")
        .select(
            "store_name",
            "product_name",
            "current_stock",
            "stock_status",
            col("event_time").alias("last_reported")
        )
        # Get only the latest status for each store/product combo
        .groupBy("store_name", "product_name")
        .agg(
            first("current_stock").alias("current_stock"),
            first("stock_status").alias("stock_status"),
            _max("last_reported").alias("alert_time")
        )
    )

---
## Executive KPI Summary

In [0]:
@dlt.table(
    name="maven_catalog.gold_schema.kpi_executive_summary",
    comment="High-level Year-over-Year KPIs — built from Gold Facts",
    table_properties={"quality": "gold"}
)
def kpi_executive_summary():
    sales = dlt.read("fact_sales")
    returns = dlt.read("fact_returns")
    calendar = dlt.read("dim_calendar")

    sales_kpi = (
        sales
        .join(calendar, sales.transaction_date == calendar.date_key, "left")
        .groupBy("year")
        .agg(
            _sum("total_revenue").alias("total_revenue"),
            _sum("total_cost").alias("total_cost"),
            _sum("total_profit").alias("total_profit"),
            _sum("quantity").alias("total_units_sold"),
            countDistinct("customer_id").alias("active_customers"),
            count("*").alias("total_transactions")
        )
    )

    returns_kpi = (
        returns
        .join(calendar, returns.return_date == calendar.date_key, "left")
        .groupBy("year")
        .agg(_sum("quantity").alias("total_units_returned"))
    )

    return (
        sales_kpi
        .join(returns_kpi, "year", "left")
        .withColumn("total_units_returned", coalesce(col("total_units_returned"), lit(0)))
        .withColumn("profit_margin_pct", spark_round((col("total_profit") / col("total_revenue")) * 100, 2))
    )

In [0]:
@dlt.table(
    name="maven_catalog.gold_schema.gold_pipeline_health_audit",
    comment="Audit table showing record counts, dropped/quarantined records, and source lineage",
    table_properties={"quality": "gold"}
)
def gold_pipeline_health_audit():
    # Helper to count records from various stages
    def get_stats(table_name, source_name, clean_ref, quarantine_ref):
        clean_count = dlt.read(clean_ref).count()
        # Handle cases where quarantine tables might be empty or missing
        try:
            quarantine_count = dlt.read(quarantine_ref).count()
        except:
            quarantine_count = 0
            
        total = clean_count + quarantine_count
        
        return spark.createDataFrame([(
            table_name, 
            source_name, 
            total, 
            clean_count, 
            quarantine_count,
            round((quarantine_count / total * 100), 2) if total > 0 else 0.0
        )], ["target_table", "silver_source", "total_records", "passed_records", "dropped_records", "failure_rate_pct"])

    # List of your major Gold tables and their Silver sources
    # Note: We use the function names defined in your previous notebooks
    stats_df = (
        get_stats("fact_sales", "slv_transactions", "fact_sales", "quarantine_transactions")
        .union(get_stats("fact_returns", "slv_returns", "fact_returns", "quarantine_returns"))
        .union(get_stats("dim_products", "slv_products", "dim_products", "quarantine_products"))
        .union(get_stats("agg_daily_sales", "fact_sales", "agg_daily_sales", "quarantine_transactions"))
    )

    return stats_df.withColumn("_audit_timestamp", current_timestamp())

---
## Summary

### Star Schema
```
                    dim_calendar
                        │
dim_customers ──── fact_sales ──── dim_products
                        │
                    dim_stores

                   fact_returns
                    │       │
              dim_stores  dim_products
```

### Tables Created (12)
| Type | Table | Key Metrics |
|---|---|---|
| Dimension | `dim_customers` | SCD2 current snapshot |
| Dimension | `dim_products` | Price tier, margin |
| Dimension | `dim_stores` | Region, age, sqft |
| Dimension | `dim_calendar` | Fiscal year, weekend flags |
| Fact | `fact_sales` | Revenue, cost, profit per transaction |
| Fact | `fact_returns` | Revenue/cost loss per return |
| Aggregate | `agg_daily_sales` | Daily by store |
| Aggregate | `agg_monthly_sales` | Monthly by store + brand |
| Aggregate | `agg_customer_lifetime` | CLV, segments |
| Aggregate | `agg_product_performance` | Sales vs returns, net profit |
| Aggregate | `agg_store_performance` | Revenue/sqft, return rate |
| KPI | `kpi_executive_summary` | Yearly top-level metrics |

### Pipeline Setup
- **Pipeline name:** `maven_gold_pipeline`
- **Source code:** This notebook
- **Target catalog:** `maven_market_uc`
- **Target schema:** `gold`

### Prerequisites
1. Gold schema must exist: `CREATE SCHEMA IF NOT EXISTS maven_market_uc.gold MANAGED LOCATION 'abfss://gold@sgmavenmarkets.dfs.core.windows.net/'`
2. External location for gold container must exist
3. All Silver tables must be populated first